In [ ]:
# New Conquer Offline — XAPK uploader
# One-cell workflow: mount Google Drive, find XAPK files, select one, and upload it as a GitHub Release asset.
from google.colab import drive, userdata
from IPython.display import display, Markdown
import os, glob, requests, json, getpass, pathlib, time

display(Markdown("## New Conquer Offline — XAPK upload"))

drive.mount('/content/drive', force_remount=False)
root = '/content/drive/MyDrive'
matches = []
for base, dirs, files in os.walk(root):
    dirs[:] = [d for d in dirs if d not in ('.Trash-1000', '.ipynb_checkpoints')]
    for name in files:
        if name.lower().endswith('.xapk'):
            matches.append(os.path.join(base, name))

if not matches:
    raise FileNotFoundError("لم يتم العثور على أي ملف XAPK داخل Google Drive/MyDrive. انقل الملف إلى MyDrive ثم أعد تشغيل الخلية.")

print("\nملفات XAPK الموجودة في Google Drive:")
for i, p in enumerate(matches, 1):
    size = os.path.getsize(p) / (1024**3)
    print(f"{i}. {p}  ({size:.2f} GB)")

while True:
    raw = input("\nاكتب رقم الملف المطلوب رفعه: ").strip()
    if raw.isdigit() and 1 <= int(raw) <= len(matches):
        selected = matches[int(raw)-1]
        break
    print("اختيار غير صحيح، حاول مرة أخرى.")

file_size = os.path.getsize(selected)
print(f"\nتم اختيار: {selected}")
print(f"الحجم: {file_size:,} bytes")

# Add a GitHub Personal Access Token in Colab > Secrets with the name GH_TOKEN.
try:
    token = userdata.get('GH_TOKEN')
except Exception:
    token = None
if not token:
    token = getpass.getpass("أدخل GitHub token (لن يظهر أثناء الكتابة): ").strip()
if not token:
    raise ValueError("لم يتم إدخال GitHub token.")

owner, repo = "ZzwahwahzZ", "New-Conquer-Offline"
api = "https://api.github.com"
headers = {"Authorization": f"Bearer {token}", "Accept": "application/vnd.github+json", "X-GitHub-Api-Version": "2022-11-28"}

# Verify repository access before creating a release.
check = requests.get(f"{api}/repos/{owner}/{repo}", headers=headers, timeout=30)
check.raise_for_status()
repo_info = check.json()
if repo_info.get("full_name", "").lower() != f"{owner}/{repo}".lower():
    raise RuntimeError("المستودع الذي تم الوصول إليه لا يطابق المستودع المطلوب.")

tag = "source-xapk"
release = requests.get(f"{api}/repos/{owner}/{repo}/releases/tags/{tag}", headers=headers, timeout=30)
if release.status_code == 404:
    created = requests.post(
        f"{api}/repos/{owner}/{repo}/releases",
        headers=headers,
        json={"tag_name": tag, "name": "Original XAPK source", "body": "Original XAPK source package for New Conquer Offline.", "draft": False, "prerelease": False},
        timeout=60
    )
    created.raise_for_status()
    release_data = created.json()
elif release.ok:
    release_data = release.json()
else:
    release.raise_for_status()

# Avoid duplicate asset names: if an asset with the same name exists, stop rather than overwrite silently.
filename = pathlib.Path(selected).name
assets = release_data.get("assets", [])
existing = [a for a in assets if a.get("name") == filename]
if existing:
    raise RuntimeError(f"يوجد بالفعل ملف بنفس الاسم في الإصدار: {filename}. لم يتم استبداله لتجنب حذف نسخة موجودة.")

upload_url = release_data["upload_url"].split("{")[0] + "?name=" + requests.utils.quote(filename)
upload_headers = {"Authorization": f"Bearer {token}", "Accept": "application/vnd.github+json", "Content-Type": "application/octet-stream"}
with open(selected, "rb") as f:
    uploaded = requests.post(upload_url, headers=upload_headers, data=f, timeout=(60, 60*60))
uploaded.raise_for_status()
asset = uploaded.json()

# Verify the uploaded release asset exists and its size matches the source.
verify = requests.get(f"{api}/repos/{owner}/{repo}/releases/{release_data['id']}", headers=headers, timeout=30)
verify.raise_for_status()
verified_assets = verify.json().get("assets", [])
verified = next((a for a in verified_assets if a.get("id") == asset.get("id")), None)
if not verified or verified.get("size") != file_size:
    raise RuntimeError("تم إرسال طلب الرفع لكن تعذر تأكيد تطابق حجم الملف على GitHub.")

print("\n✅ تم رفع الملف والتحقق من حجمه.")
print("اسم الملف:", verified["name"])
print("الحجم:", f"{verified['size']:,}", "bytes")
print("رابط التنزيل:", verified["browser_download_url"])
print("رابط الإصدار:", release_data["html_url"])
